In [1]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, RocCurveDisplay, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score
)
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings("ignore")

df = pd.read_excel("data/data.xlsx", index_col=0)
targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]

X = df[features]
y = (df["SI"] > 8).astype(int).values
print(f"Класс 1 (SI > 8): {y.sum()} ({y.mean()*100:.1f}%)")
print(f"Класс 0 (SI ≤ 8): {(1-y).sum()} ({(1-y).mean()*100:.1f}%)")
print("Дисбаланс: необходима обработка!")

# Заполняем NaN медианой
X = X.fillna(X.median())

# Удаление нулевой дисперсии
selector = VarianceThreshold(threshold=0.0)
X = selector.fit_transform(X)
feat_names = [features[i] for i in selector.get_support(indices=True)]
print(f"Признаков: {X.shape[1]}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Класс 1 (SI > 8): 357 (35.7%)
Класс 0 (SI ≤ 8): 644 (64.3%)
Дисбаланс: необходима обработка!
Признаков: 192


## Базовые модели без балансировки

In [2]:
print("\n── Базовые модели (без балансировки) ──")
base_models = {
    "LogisticRegression": Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=1000))]),
    "RF (class_weight)":  RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                                  random_state=42, n_jobs=-1),
    "GB":                 GradientBoostingClassifier(n_estimators=100, random_state=42),
}
for name, model in base_models.items():
    auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc").mean()
    f1  = cross_val_score(model, X, y, cv=cv, scoring="f1").mean()
    print(f"  {name:28s}: AUC={auc:.3f}  F1={f1:.3f}")


── Базовые модели (без балансировки) ──
  LogisticRegression          : AUC=0.684  F1=0.533
  RF (class_weight)           : AUC=0.728  F1=0.578
  GB                          : AUC=0.724  F1=0.535


## SMOTE для балансировки

In [3]:
print("\n── SMOTE-балансировка ──")
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)
print(f"После SMOTE: {X_res.shape[0]} образцов, класс 1: {y_res.sum()}")


── SMOTE-балансировка ──
После SMOTE: 1288 образцов, класс 1: 644


## GridSearchCV на сбалансированных данных

In [4]:
print("\n── GridSearchCV: LogisticRegression (SMOTE) ──")
gs_lr = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=2000))]),
    {"m__C": [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_lr.fit(X_res, y_res)
print(f"  Best: {gs_lr.best_params_}  AUC={gs_lr.best_score_:.4f}")

print("\n── GridSearchCV: RandomForest (SMOTE) ──")
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_rf.fit(X_res, y_res)
print(f"  Best: {gs_rf.best_params_}  AUC={gs_rf.best_score_:.4f}")

print("\n── GridSearchCV: GradientBoosting (SMOTE) ──")
gs_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1, 0.2], "max_depth": [3, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_gb.fit(X_res, y_res)
print(f"  Best: {gs_gb.best_params_}  AUC={gs_gb.best_score_:.4f}")


── GridSearchCV: LogisticRegression (SMOTE) ──
  Best: {'m__C': 10}  AUC=0.7626

── GridSearchCV: RandomForest (SMOTE) ──
  Best: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}  AUC=0.8394

── GridSearchCV: GradientBoosting (SMOTE) ──
  Best: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}  AUC=0.8432


## GridSearchCV: RF с class_weight на исходных данных

In [5]:
print("\n── GridSearchCV: RandomForest (class_weight=balanced) ──")
gs_rf_cw = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_rf_cw.fit(X, y)
print(f"  Best: {gs_rf_cw.best_params_}  AUC={gs_rf_cw.best_score_:.4f}")


── GridSearchCV: RandomForest (class_weight=balanced) ──
  Best: {'max_depth': None, 'n_estimators': 200}  AUC=0.7293


## Итог

In [6]:
print("\n── Итоговое сравнение ROC-AUC ──")
tuned_auc = {
    "LR (SMOTE)":           gs_lr.best_score_,
    "RF (SMOTE)":           gs_rf.best_score_,
    "GB (SMOTE)":           gs_gb.best_score_,
    "RF (class_weight)":    gs_rf_cw.best_score_,
}
for name, auc in sorted(tuned_auc.items(), key=lambda x: -x[1]):
    print(f"  {name:28s}: AUC = {auc:.4f}")


── Итоговое сравнение ROC-AUC ──
  GB (SMOTE)                  : AUC = 0.8432
  RF (SMOTE)                  : AUC = 0.8394
  LR (SMOTE)                  : AUC = 0.7626
  RF (class_weight)           : AUC = 0.7293


## ROC и PR кривые

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("SI > 8: ROC и Precision-Recall кривые")

for name, gs, X_eval, y_eval in [
    ("LR (SMOTE)",        gs_lr,    X_res, y_res),
    ("RF (SMOTE)",        gs_rf,    X_res, y_res),
    ("GB (SMOTE)",        gs_gb,    X_res, y_res),
    ("RF (class_weight)", gs_rf_cw, X,     y),
]:
    RocCurveDisplay.from_estimator(gs.best_estimator_, X_eval, y_eval, ax=axes[0], name=name)

axes[0].set_title("ROC")

# PR-кривые (более информативны при дисбалансе)
best_gb = gs_gb.best_estimator_
y_scores = best_gb.predict_proba(X_res)[:, 1]
precision, recall, _ = precision_recall_curve(y_res, y_scores)
ap = average_precision_score(y_res, y_scores)
axes[1].plot(recall, precision, label=f"GB (AP={ap:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall (GradientBoosting, SMOTE)")
axes[1].legend()

plt.tight_layout()
plt.savefig("plots/cls_si8_roc_pr.png")
plt.close()
print("\nСохранён график: cls_si8_roc_pr.png")


Сохранён график: cls_si8_roc_pr.png


## Матрица ошибок

In [8]:
best_gs = max([gs_rf, gs_gb, gs_lr, gs_rf_cw], key=lambda g: g.best_score_)
try:
    y_pred = best_gs.predict(X_res)
    y_true = y_res
except Exception:
    y_pred = best_gs.predict(X)
    y_true = y
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False)
ax.set_title("Матрица ошибок (лучшая модель, SI > 8)")
plt.tight_layout()
plt.savefig("plots/cls_si8_cm.png")
plt.close()
print("Сохранён график: cls_si8_cm.png")
print("\nClassification Report (лучшая модель):")
print(classification_report(y_true, y_pred, target_names=["SI ≤ 8", "SI > 8"]))

Сохранён график: cls_si8_cm.png

Classification Report (лучшая модель):
              precision    recall  f1-score   support

      SI ≤ 8       0.97      0.94      0.95       644
      SI > 8       0.94      0.97      0.95       644

    accuracy                           0.95      1288
   macro avg       0.95      0.95      0.95      1288
weighted avg       0.95      0.95      0.95      1288


── Выводы ──────────────────────────────────────────────────────────────────────
1. Задача несбалансирована: 644 (SI ≤ 8) vs 357 (SI > 8), соотношение ≈ 1.8:1.
   SMOTE балансирует классы для обучения.
2. На ROC с SMOTE: GB — AUC=0.99, RF — AUC=0.98, значительно лучше LR (AUC=0.87).
   Precision-Recall кривая: AP=0.992 для GB — почти идеальное разделение на
   сбалансированных данных после SMOTE.
3. Матрица ошибок (лучшая модель): 604+624=1228 правильных, 40+20=60 ошибок
   из 1288 (accuracy=0.953 на SMOTE-данных). Важно: 20 FN (пропущенных SI > 8)
   против 40 FP — модель немного консервативн

## Итоговые выводы

Вывод по классификации SI > 8 (хорошая селективность)

- Исходный дисбаланс: класс 1 (SI > 8) составляет 35.7% (357 образцов), класс 0 — 64.3% (644 образца). Без обработки дисбаланса модели показывают низкое качество (AUC ≤ 0.73, F1 ≤ 0.58).

- Применение SMOTE (синтетическая балансировка классов) значительно улучшает результаты. После SMOTE оба класса содержат по 644 образца.

- Лучший результат достигается у GradientBoosting с SMOTE: AUC = 0.8432. Параметры: learning_rate=0.1, max_depth=5, n_estimators=100.

- RandomForest с SMOTE даёт близкий результат: AUC = 0.8394 (max_depth=10, min_samples_split=5, n_estimators=200).

- LogisticRegression с SMOTE заметно слабее (AUC = 0.7626), что указывает на нелинейный характер зависимости.

- Использование class_weight='balanced' без SMOTE (RandomForest) даёт лишь AUC = 0.7293 — существенно хуже, чем с SMOTE.

- Итоговый classification report для лучшей модели (GradientBoosting с SMOTE) на сбалансированной выборке показывает отличное качество: precision=0.94-0.97, recall=0.94-0.97, accuracy=0.95, macro avg F1=0.95.

- Таким образом, предсказание «хорошей селективности» (SI > 8) после правильной балансировки классов становится весьма точным — значительно лучше, чем предсказание селективности относительно медианы (SI > 3.85, где AUC был около 0.73). Это объясняется тем, что крайне селективные соединения (SI > 8) обладают более чёткими структурными отличиями.